#Esercitazione 1 — Dalle regole scritte a mano agli alberi di decisione

**new ALF – Tutorato** · Corso di Laurea in Informatica · AA 2026/27

Nelle lezioni abbiamo visto la differenza tra **programmazione classica** (scriviamo noi le regole) e **apprendimento automatico** (le regole vengono ricavate dai dati), come si **valuta** un classificatore (matrice di confusione, accuracy, precision, recall, F1) e un primo modello di ML, l'**albero di decisione** del dataset *Play Tennis*.

Oggi mettiamo le mani su tutte queste cose.

### Come si usa questo notebook
- Ogni **cella** si esegue con **Shift + Invio** (o con il tasto ▶ a sinistra). Esegui le celle **in ordine**, dall'alto verso il basso.
- Le celle segnate con **✏️** sono esercizi: modifica il codice dove indicato.
- Sotto ogni esercizio c'è una cella **💡 Soluzione** con il codice nascosto. Prova prima da solo! Se resti bloccato puoi **eseguirla** (▶) per proseguire con la soluzione, oppure aprirla con un doppio clic per leggerla.
- ⚠️ Eseguire una cella Soluzione **sostituisce il tuo codice** con il suo: se vuoi continuare con la tua versione, saltala.
- Se qualcosa si rompe: menu **Runtime → Riavvia sessione** (*Restart session*) e riesegui le celle dall'alto.

In [2]:
#@title 📦 Installazione: esegui questa cella per prima (installa le librerie necessarie)
# %pip installa le librerie nell'ambiente in cui gira il notebook; -q riduce i messaggi a schermo.
# Su Colab sono già presenti (la cella finisce in pochi secondi), in locale vengono scaricate.
%pip install -q pandas numpy scikit-learn matplotlib

In [3]:
#@title ⚙️ Setup: esegui questa cella subito dopo l'installazione (crea i file con i dati)
# Questa cella scrive nel workspace di Colab il file "commenti_train.csv"
# con i commenti di esempio per la Parte 1. Puoi vederlo nel pannello 📁 a sinistra.
# Nota: il workspace di Colab si svuota quando la sessione termina,
# quindi questa cella va rieseguita ogni volta che riapri il notebook.

TRAIN_CSV = """testo,etichetta
"cibo ottimo e personale gentilissimo, torneremo sicuramente",positivo
"pizza buonissima, impasto leggero e ingredienti freschi",positivo
"locale accogliente e prezzi onesti, lo consiglio a tutti",positivo
"esperienza fantastica, il tiramisù era perfetto",positivo
"servizio veloce e cameriere simpatici, ottimo rapporto qualità prezzo",positivo
"tutto perfetto, dalla carbonara al dolce",positivo
"porzioni abbondanti e piatti buonissimi, consigliato",positivo
"cena splendida in un posto bellissimo, grazie di tutto",positivo
il pesce era freschissimo e cucinato benissimo,positivo
"titolare gentile e disponibile, ci siamo trovati benissimo",positivo
ottima scelta di vini e atmosfera rilassante,positivo
bravi! tornerò presto con gli amici,positivo
"cibo pessimo e cameriere scortese, mai più",negativo
"pizza bruciata e fredda, una delusione",negativo
prezzi troppo alti per porzioni minuscole,negativo
abbiamo aspettato un'ora per un piatto di pasta scotta,negativo
"locale sporco e rumoroso, non lo consiglio",negativo
servizio lentissimo e personale maleducato,negativo
la carne era dura e il vino cattivo,negativo
"esperienza deludente, non mi è piaciuto niente",negativo
"conto sbagliato e nessuno si è scusato, pessimo",negativo
tavolo sporco e bagno inguardabile,negativo
"dolce industriale spacciato per fatto in casa, che delusione",negativo
"mai mangiato così male, sconsigliato",negativo
"""

with open("commenti_train.csv", "w", encoding="utf-8") as f:
    f.write(TRAIN_CSV)

print("✅ File commenti_train.csv creato!")

✅ File commenti_train.csv creato!


---
# Parte 0 — Python in 10 minuti

Python è il linguaggio più usato nel machine learning. Qui vediamo **solo** gli strumenti che ci servono oggi. Esegui ogni cella, leggi i commenti (le righe che iniziano con `#`, che Python ignora) e prova a cambiare qualcosa per vedere cosa succede.

### 0.1 Variabili e stringhe

In [ ]:
commento = "Il Cibo era OTTIMO"   # assegno una stringa alla variabile commento

# print() mostra a schermo uno o più valori, separati da virgole
print(commento)

# .lower() è un "metodo" delle stringhe: restituisce una copia tutta in minuscolo
print(commento.lower())

# len() restituisce la lunghezza: per una stringa, il numero di caratteri
print("Numero di caratteri:", len(commento))

# f-string: con la f davanti alle virgolette, ciò che sta tra { } viene sostituito dal suo valore
voto = 27.456
print(f"Il commento è: {commento} | voto: {voto:.1f}")   # :.1f = mostra 1 cifra decimale

### 0.2 L'operatore `in`: "questa parola compare nel testo?"
`a in b` vale **`True`** (vero) se la stringa `a` compare dentro `b`, altrimenti **`False`** (falso). `True` e `False` sono i due valori **booleani** di Python.

In [ ]:
testo = "pizza buonissima e servizio veloce"

print("buonissima" in testo)   # True
print("pessimo" in testo)      # False
print("pessimo" not in testo)  # not in è il contrario di in: True

# ⚠️ Attenzione: in cerca SOTTOSTRINGHE (sequenze di caratteri), non parole intere!
print("ottim" in "una cena ottima")                 # True: un pezzo di parola cattura ottimo, ottima, ottimi... utile!
print("male" in "un locale che ama ogni animale")   # True: ma qui "male" non c'entra niente!
print("lento" in "servizio lentissimo")             # False: "lentissimo" non contiene la sequenza "lento"

### 0.3 Decisioni: `if` / `elif` / `else`
Python esegue un blocco di codice solo se la condizione è vera. **L'indentazione è obbligatoria**: indica quali righe stanno "dentro" l'`if`. Le condizioni si combinano con `and` (entrambe vere), `or` (almeno una vera) e `not` (negazione).

In [ ]:
testo = "pizza buonissima ma servizio lento"

if "buonissima" in testo and "lento" in testo:      # and: entrambe le condizioni devono essere vere
    print("commento misto")
elif "buonissima" in testo or "ottima" in testo:    # elif = "altrimenti, se...": controllato solo se l'if è falso
    print("commento positivo")
else:                                               # else: se nessuna condizione precedente è vera
    print("non so decidere")

### 0.4 Liste e cicli `for`
Una **lista** contiene più valori in ordine, tra parentesi quadre `[ ]`. Il ciclo **`for`** ripete un blocco di codice una volta per ogni elemento della lista.

In [ ]:
parole_positive = ["ottimo", "buonissimo", "perfetto"]   # una lista di 3 stringhe

print(parole_positive[0])      # gli indici partono da 0: questo è il primo elemento
print(len(parole_positive))    # len() su una lista restituisce il numero di elementi

parole_positive.append("squisito")   # .append() aggiunge un elemento in fondo alla lista
print(parole_positive)

testo = "tutto perfetto e il dolce era ottimo"
conteggio = 0                        # un contatore che parte da zero
for parola in parole_positive:       # a ogni giro, la variabile parola contiene un elemento diverso
    if parola in testo:
        conteggio = conteggio + 1    # forma abbreviata equivalente: conteggio += 1
print("Parole positive trovate:", conteggio)

### 0.5 Funzioni
Una **funzione** è un pezzo di codice riutilizzabile: si definisce con `def`, riceve dei **parametri** e restituisce un risultato con `return`.

In [ ]:
def conta_parole(testo, lista_parole):
    """Restituisce quante parole di lista_parole compaiono in testo."""   # docstring: descrive la funzione
    conteggio = 0
    for parola in lista_parole:
        if parola in testo:
            conteggio += 1
    return conteggio          # return restituisce il valore a chi ha chiamato la funzione

risultato = conta_parole("tutto perfetto e ottimo", ["ottimo", "perfetto", "pessimo"])
print(risultato)

### 0.6 Due strumenti che useremo dopo: `zip` e i dizionari

In [ ]:
# zip() scorre due (o più) liste IN PARALLELO: a ogni giro prende un elemento da ciascuna
nomi = ["anna", "luca", "sara"]
voti = [28, 30, 25]
for nome, voto in zip(nomi, voti):
    print(nome, "ha preso", voto)

# Un dizionario (dict) associa CHIAVI a VALORI: {chiave: valore, ...}
capitali = {"italia": "roma", "francia": "parigi"}
print(capitali["italia"])            # accedo al valore tramite la chiave
capitali["spagna"] = "madrid"        # aggiungo una nuova coppia chiave-valore
print("germania" in capitali)        # in su un dizionario controlla le CHIAVI: False
print(capitali.get("germania", "?")) # .get(chiave, default): se la chiave non c'è restituisce il default

---
# Parte 1 — Sei tu l'algoritmo: sentiment analysis a regole

Riprendiamo l'esercizio della lezione 2: **scrivere un programma che decida se un commento è positivo o negativo**.

Lo facciamo come nella programmazione classica, cioè scrivendo noi le regole con `if`/`else`. Però ragioniamo come nel machine learning:

1. guardiamo dei **commenti di esempio** già etichettati: questo è il nostro **training set**;
2. scriviamo le regole guardando quegli esempi. **In pratica, l'algoritmo di apprendimento sei tu**;
3. misuriamo quanto funzionano le regole, con le metriche viste a lezione;
4. alla fine le proviamo su **commenti nuovi, mai visti**: il **test set**.

### 1.1 Carichiamo i commenti di esempio
Per leggere i dati usiamo **pandas**, la libreria Python più usata per lavorare con tabelle. Una tabella pandas si chiama **DataFrame**.

In [ ]:
import pandas as pd    # import carica una libreria; "as pd" le dà un soprannome più corto

# read_csv legge un file CSV (valori separati da virgole) e lo trasforma in un DataFrame
train = pd.read_csv("commenti_train.csv")

# se l'ultima riga di una cella è un'espressione, Colab ne mostra il valore: qui la tabella
train

In [ ]:
# train["etichetta"] seleziona una colonna; .value_counts() conta quante volte compare ogni valore
train["etichetta"].value_counts()

In [ ]:
# .tolist() converte una colonna del DataFrame in una normale lista Python
testi_train = train["testo"].tolist()
# Trasformiamo le etichette in booleani: True = commento positivo, False = commento negativo
etichette_train = (train["etichetta"] == "positivo").tolist()

print(len(testi_train), "commenti nel training set")
print("Esempio:", testi_train[0], "→", etichette_train[0])

### 1.2 ✏️ Esercizio: scrivi le tue regole
Completa la funzione `classifica(testo)`: deve restituire **`True`** se il commento è positivo e **`False`** se è negativo. **I testi sono già tutti in minuscolo.**

💡 *Suggerimento*: crea una lista di parole positive e una di parole negative, conta quante ne compaiono nel testo (come nella Parte 0) e decidi in base al conteggio. Guarda bene la tabella dei commenti qui sopra!

In [ ]:
def classifica(testo):
    # ✏️ scrivi qui le tue regole
    return True

In [ ]:
#@title 💡 Soluzione 1.2 — doppio clic per leggerla, ▶ per usarla (sostituisce il tuo codice!)
# Usare pezzi di parola (es. "ottim") permette di catturare ottimo, ottima, ottimi.
# Vedremo però che ha anche dei rischi…
def classifica(testo):
    # parole (o pezzi di parole) che ho visto nei commenti positivi del training set
    positive = ["ottim", "buonissim", "perfett", "consiglio", "consigliato", "gentil",
                "fantastic", "benissimo", "torner", "fresc", "splendid", "bravi",
                "simpatic", "accogliente"]
    # ...e in quelli negativi
    negative = ["pessim", "scortese", "mai più", "delusione", "deludente", "sporco",
                "lent", "maleducat", "cattiv", "sconsigliato", "fredd", "scott",
                "bruciat", "troppo", "male", "niente"]

    punteggio = 0
    for parola in positive:
        if parola in testo:
            punteggio += 1
    for parola in negative:
        if parola in testo:
            punteggio -= 1

    return punteggio >= 0     # True se positivo; in caso di pareggio scelgo True

Ora applichiamo `classifica` a tutti i commenti e vediamo dove sbaglia.

In [ ]:
# "list comprehension": crea una lista applicando classifica() a ogni testo.
# È equivalente a:
#   predizioni_train = []
#   for t in testi_train:
#       predizioni_train.append(classifica(t))
predizioni_train = [classifica(t) for t in testi_train]

for testo, vero, predetto in zip(testi_train, etichette_train, predizioni_train):
    simbolo = "✅" if vero == predetto else "❌"        # if/else scritto su una sola riga
    print(f"{simbolo} vero={str(vero):<5} predetto={str(predetto):<5} | {testo}")   # :<5 = allinea a sinistra su 5 caratteri

### 1.3 ✏️ Esercizio: calcoliamo le metriche
Ricordiamo la **matrice di confusione** vista a lezione (classe positiva = `True`):

| | **vero positivo** | **vero negativo** |
|---|---|---|
| **predetto positivo** | TP (true positive) | FP (false positive) |
| **predetto negativo** | FN (false negative) | TN (true negative) |

$$\text{accuracy} = \frac{TP+TN}{TP+FP+TN+FN} \qquad \text{precision} = \frac{TP}{TP+FP} \qquad \text{recall} = \frac{TP}{TP+FN} \qquad F_1 = \frac{2 \cdot P \cdot R}{P+R}$$

Completa le due funzioni. Il parametro `positivo=True` ha un **valore di default**: se non lo specifichiamo, vale `True`. Ci servirà per riusare le funzioni nella Parte 2 con altre etichette.

⚠️ Se un denominatore è 0 (es. il classificatore non predice mai `True`), la divisione dà errore: in quel caso restituisci 0.

In [ ]:
def matrice_confusione(veri, predetti, positivo=True):
    tp = 0
    fp = 0
    fn = 0
    tn = 0
    for v, p in zip(veri, predetti):
        if v == positivo and p == positivo:
            tp += 1
        # ✏️ completa gli altri tre casi (fp, fn, tn) con elif / else
    return tp, fp, fn, tn     # una funzione può restituire più valori separati da virgola

In [ ]:
#@title 💡 Soluzione 1.3 — doppio clic per leggerla, ▶ per usarla (sostituisce il tuo codice!)
def matrice_confusione(veri, predetti, positivo=True):
    tp = 0
    fp = 0
    fn = 0
    tn = 0
    for v, p in zip(veri, predetti):
        if v == positivo and p == positivo:
            tp += 1          # era positivo e l'ho detto positivo
        elif v != positivo and p == positivo:
            fp += 1          # era negativo ma l'ho detto positivo
        elif v == positivo and p != positivo:
            fn += 1          # era positivo ma l'ho detto negativo
        else:
            tn += 1          # era negativo e l'ho detto negativo
    return tp, fp, fn, tn

In [ ]:
def calcola_metriche(veri, predetti, positivo=True):
    tp, fp, fn, tn = matrice_confusione(veri, predetti, positivo)
    accuracy = (tp + tn) / (tp + fp + tn + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

Questa funzione di supporto stampa tutto in modo ordinato. Non serve modificarla.

In [ ]:
def stampa_metriche(titolo, veri, predetti, positivo=True):
    tp, fp, fn, tn = matrice_confusione(veri, predetti, positivo)
    metriche = calcola_metriche(veri, predetti, positivo)
    print(f"--- {titolo} ---")
    print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")
    for nome, valore in metriche.items():      # .items() scorre le coppie (chiave, valore) del dizionario
        print(f"{nome:>10}: {valore:.2f}")       # :>10 allinea a destra su 10 caratteri, :.2f = 2 decimali
    print()

stampa_metriche("Le mie regole – TRAINING SET", etichette_train, predizioni_train)

### 1.4 Verifichiamo con scikit-learn
**scikit-learn** (`sklearn`) è la libreria Python di riferimento per il machine learning "classico". Tra le altre cose contiene, nel modulo `sklearn.metrics`, tutte le metriche viste a lezione. Usiamola per controllare che i nostri calcoli siano giusti.

In [ ]:
# from ... import ... importa solo alcune funzioni da un modulo della libreria
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# pos_label indica qual è la classe "positiva"; zero_division=0 evita errori se un denominatore è 0
print("accuracy :", accuracy_score(etichette_train, predizioni_train))
print("precision:", precision_score(etichette_train, predizioni_train, pos_label=True, zero_division=0))
print("recall   :", recall_score(etichette_train, predizioni_train, pos_label=True, zero_division=0))
print("f1       :", f1_score(etichette_train, predizioni_train, pos_label=True, zero_division=0))

# labels fissa l'ordine delle classi nella matrice
print(confusion_matrix(etichette_train, predizioni_train, labels=[True, False]))

> ⚠️ **Attenzione all'orientamento!** In scikit-learn le **righe** della matrice di confusione sono i valori **veri** e le **colonne** quelli **predetti**: è il contrario della tabella delle slide. Quindi qui la prima riga è `[TP, FN]` e la seconda `[FP, TN]`. Controlla sempre la documentazione!

### 1.5 Il momento della verità: il test set
Le tue regole funzionano bene sugli esempi che hai guardato. Ma un sistema di sentiment analysis serve per i commenti **che arriveranno domani**, non per quelli che hai già letto.

Esegui la cella qui sotto per sbloccare **24 commenti nuovi**. 🚫 **Regola del gioco**: da ora in poi **non puoi più modificare** `classifica()` guardando questi commenti. Se lo fai, il test set non misura più niente, perché diventa un secondo training set: è uno degli *errori classici* visti a lezione.

In [ ]:
#@title 🔒 Sblocca il test set (esegui la cella)
# I commenti sono codificati per non essere letti per sbaglio prima del tempo.
import base64
TEST_B64 = "dGVzdG8sZXRpY2hldHRhCnNxdWlzaXRvISBjb21wbGltZW50aSBhbGxvIGNoZWYscG9zaXRpdm8KbGFzYWduZSBlY2NlbGxlbnRpIGUgc2Vydml6aW8gaW1wZWNjYWJpbGUscG9zaXRpdm8KImRhdnZlcm8gb3R0aW1vLCBjaSB0b3JuZXJlbW8gZGkgc2ljdXJvIixwb3NpdGl2bwoibm9uIGVyYSBhZmZhdHRvIG1hbGUsIGFuemkgY2kgc2lhbW8gZGl2ZXJ0aXRpIixwb3NpdGl2bwpwZXJzb25hbGUgY29yZGlhbGUgZSBjdWNpbmEgZGVsaXppb3NhLHBvc2l0aXZvCiJwcmV6emkgdW4gcG8nIGFsdGkgbWEgbmUgdmFsZSBsYSBwZW5hLCB0dXR0byBidW9uaXNzaW1vIixwb3NpdGl2bwp1bmEgc2NvcGVydGEgbWVyYXZpZ2xpb3NhIG5lbCBjZW50cm8gZGkgY2FnbGlhcmkscG9zaXRpdm8KZ25vY2NoaSBmYXR0aSBpbiBjYXNhIHN0cmVwaXRvc2kscG9zaXRpdm8KY29uc2lnbGlvIHZpdmFtZW50ZSBsYSBncmlnbGlhdGEgZGkgcGVzY2UscG9zaXRpdm8KcG9zdG8gY2FyaW5vIGUgY2FtZXJpZXJpIGdlbnRpbGkscG9zaXRpdm8KIm5vbiBobyBuaWVudGUgZGEgY3JpdGljYXJlLCBwZXJmZXR0byIscG9zaXRpdm8KbWVuw7kgZGVsIHByYW56byBlY29ub21pY28gZSBhYmJvbmRhbnRlLHBvc2l0aXZvCiJjaWJvIGltbWFuZ2lhYmlsZSwgZGEgZXZpdGFyZSIsbmVnYXRpdm8KImNoZSBzY2hpZm8sIHBhc3RhIGNvbGxvc2EgZSBzdWdvIGluc2lwaWRvIixuZWdhdGl2bwoiaWwgY2libyBlcmEgb3R0aW1vIG1hIGlsIHNlcnZpemlvIHBlc3NpbW8sIG5vbiB0b3JuZXJlbW8iLG5lZ2F0aXZvCiJjb21wbGltZW50aSwgcXVhcmFudGEgbWludXRpIHBlciB1bmEgcGl6emEgZnJlZGRhIixuZWdhdGl2bwoibmllbnRlIGRpIHNwZWNpYWxlLCBhbnppIHBpdXR0b3N0byBtZWRpb2NyZSIsbmVnYXRpdm8KImNhbWVyaWVyZSBhcnJvZ2FudGUsIGNpIGhhIHRyYXR0YXRpIG1hbGUiLG5lZ2F0aXZvCm5vbiBjb25zaWdsaW8gcXVlc3RvIHJpc3RvcmFudGUgYSBuZXNzdW5vLG5lZ2F0aXZvCiJmcml0dHVyYSB1bnRhIGUgcGVzYW50ZSwgc29ubyBzdGF0byBtYWxlIHR1dHRhIGxhIG5vdHRlIixuZWdhdGl2bwpvdHRpbW8uLi4gc2UgdGkgcGlhY2UgYXNwZXR0YXJlIHVuJ29yYSxuZWdhdGl2bwpwb3J6aW9uaSByaWRpY29sZSBlIGNvbnRvIHNhbGF0aXNzaW1vLG5lZ2F0aXZvCnJ1bW9yZSBpbnNvcHBvcnRhYmlsZSBlIHRhdm9saSBhdHRhY2NhdGksbmVnYXRpdm8KcHJpbWEgdm9sdGEgZSB1bHRpbWEgdm9sdGEsbmVnYXRpdm8K"
with open("commenti_test.csv", "w", encoding="utf-8") as f:
    f.write(base64.b64decode(TEST_B64).decode("utf-8"))
print("🔓 File commenti_test.csv creato!")

In [ ]:
test = pd.read_csv("commenti_test.csv")
testi_test = test["testo"].tolist()
etichette_test = (test["etichetta"] == "positivo").tolist()

predizioni_test = [classifica(t) for t in testi_test]

stampa_metriche("Le mie regole – TRAINING SET", etichette_train, predizioni_train)
stampa_metriche("Le mie regole – TEST SET", etichette_test, predizioni_test)

In [ ]:
# Dove sbaglia sul test set?
for testo, vero, predetto in zip(testi_test, etichette_test, predizioni_test):
    simbolo = "✅" if vero == predetto else "❌"
    print(f"{simbolo} vero={str(vero):<5} predetto={str(predetto):<5} | {testo}")

### 🗣️ Discussione
- Di quanto sono **calate** le metriche dal training al test? Perché?
- Guarda gli errori e cerca di classificarli:
  - **parole mai viste**: *squisito, immangiabile, strepitosi…*
  - **negazioni**: *"non consiglio"* contiene *"consiglio"*; *"non era affatto male"* contiene *"male"*
  - **sarcasmo**: *"complimenti, quaranta minuti per una pizza fredda"*
  - **commenti misti**: *"il cibo era ottimo ma il servizio pessimo"*
  - **sottostringhe traditrici**: ti ricordi la Parte 0? (prova `"lent" in "lasagne eccellenti"`)
- Quale metrica è peggiorata di più tra precision e recall? Cosa significa in pratica?

### 1.6 Il caso estremo: il memorizzatore
Proviamo il "modello" più pigro possibile: **impara a memoria** il training set. Se gli diamo un commento che ha già visto, risponde con l'etichetta giusta; altrimenti tira a indovinare e dice sempre `True` (positivo).

In [ ]:
memoria = {}                                   # un dizionario vuoto
for testo, etichetta in zip(testi_train, etichette_train):
    memoria[testo] = etichetta                 # chiave = commento, valore = etichetta corretta

def memorizzatore(testo):
    return memoria.get(testo, True)      # se non l'ha mai visto: True

stampa_metriche("Memorizzatore – TRAINING SET", etichette_train, [memorizzatore(t) for t in testi_train])
stampa_metriche("Memorizzatore – TEST SET", etichette_test, [memorizzatore(t) for t in testi_test])

### 📌 Overfitting (sovradattamento)
Il memorizzatore è **perfetto** sul training set e **inutile** sul test set: non ha imparato niente, ha solo memorizzato.

Si parla di **overfitting** quando un modello si adatta così tanto ai dati di training da "impararne" anche i dettagli casuali, e per questo **non generalizza** ai dati nuovi. Il segnale tipico è una **grande differenza tra le prestazioni sul training e quelle sul test**.

Con le parole della lezione 3: noi vogliamo che $\hat{f}$ assomigli alla vera $f$ su **tutti** i dati possibili, non solo su quelli che abbiamo visto. Le metriche calcolate sul test set sono il nostro modo per stimare questa somiglianza.

Anche le tue regole potrebbero essere in overfitting, se hai aggiunto parole che compaiono una sola volta nel training. Nella Parte 2 vedremo lo stesso fenomeno con un vero modello di machine learning.

---
# Parte 2 — Alberi di decisione con scikit-learn

Nella lezione 2 abbiamo visto l'albero di *Play Tennis*: era un modello appreso dai dati, ma in fondo è solo una serie di `if`/`else`:
```
se Outlook == Sunny:      se Humidity == High → No,  altrimenti → Yes
se Outlook == Overcast:   → Yes
se Outlook == Rain:       se Wind == Strong → No,    altrimenti → Yes
```
Ora facciamo lo stesso con **dati numerici**. Prima l'albero lo scrivi **tu**, poi lo facciamo costruire a **scikit-learn**, e infine li confrontiamo.

### 2.1 Il dataset Wine
Il dataset *Wine* è incluso in scikit-learn: contiene le analisi chimiche di **vini prodotti in Italia nella stessa regione da tre diverse cantine (cultivar)**. Per semplicità:
- teniamo solo **due cantine**, che chiameremo **A** e **B** (classificazione **binaria**);
- usiamo solo **due misure** (feature): la **gradazione alcolica** (`alcol`, in %) e l'**intensità del colore** (`colore`).

**Il problema:** dato un vino, misuro alcol e colore. **Da quale cantina viene?**

In [ ]:
from sklearn.datasets import load_wine     # funzione che carica il dataset Wine (non serve scaricare nulla)

vini = load_wine(as_frame=True)            # as_frame=True: restituisce i dati come DataFrame pandas
df = vini.frame                            # tabella completa: 13 misure chimiche + colonna "target" (cantina 0, 1 o 2)

df = df[df["target"].isin([0, 2])]         # filtro: tengo solo le righe in cui target è 0 oppure 2
df = df.rename(columns={"alcohol": "alcol", "color_intensity": "colore"})   # rinomino le colonne in italiano
df["cantina"] = df["target"].map({0: "A", 2: "B"})                          # map: sostituisce i valori secondo il dizionario
df = df[["alcol", "colore", "cantina"]]   # tengo solo le 3 colonne che ci interessano

print(len(df), "vini")
df.head()                                  # .head() mostra le prime 5 righe

In [ ]:
# .describe() calcola statistiche di base (media, min, max...) per ogni colonna numerica;
# con groupby("cantina") le calcola separatamente per la cantina A e per la B
df.groupby("cantina").describe().round(2)

### 2.2 Training set e test set
Questa volta non abbiamo due file già pronti: dividiamo noi i dati con `train_test_split`. Il **70%** va nel training set (lo guardiamo e lo usiamo per costruire gli alberi) e il **30%** nel test set (lo usiamo solo alla fine).

Per convenzione, nel machine learning:
- **`X`** (maiuscola) contiene le **feature**, cioè l'input del modello;
- **`y`** contiene il **target**, cioè l'output da predire.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[["alcol", "colore"]]    # con doppie parentesi quadre seleziono più colonne: un DataFrame
y = df["cantina"]              # con parentesi singole seleziono una colonna sola

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,      # 30% dei dati nel test set
    random_state=23,    # la divisione è casuale: fissare il "seme" fa sì che tutti otteniamo la stessa
    stratify=y,         # mantiene la stessa proporzione di A e B nel training e nel test set
)

print("Training set:", len(X_train), "vini")
print("Test set:    ", len(X_test), "vini")

In [ ]:
#@title 🛠️ Funzioni di supporto per i grafici (esegui la cella, non serve leggerla)
# matplotlib è la libreria standard per i grafici in Python; numpy per i calcoli numerici su array
import numpy as np
import matplotlib.pyplot as plt

COLORI = {"A": "tab:red", "B": "tab:blue"}

def predici(modello, X):
    """Predizioni per X. Funziona sia con un modello sklearn (che ha .predict)
    sia con una funzione scritta da noi del tipo f(alcol, colore)."""
    if hasattr(modello, "predict"):
        return list(modello.predict(X))
    return [modello(a, c) for a, c in zip(X["alcol"], X["colore"])]

def mostra_dati(X, y, titolo=""):
    """Grafico a dispersione (scatter plot): un punto per ogni vino, colorato per cantina."""
    plt.figure(figsize=(7, 5))
    for cantina, colore in COLORI.items():
        sel = (y == cantina).values
        plt.scatter(X["alcol"][sel], X["colore"][sel], c=colore, label=f"cantina {cantina}",
                    edgecolors="k", s=50)
    plt.xlabel("alcol (%)")
    plt.ylabel("intensità del colore")
    plt.xticks(np.arange(12, 15.01, 0.25), rotation=45)
    plt.yticks(np.arange(3, 13.01, 0.5))
    plt.grid(alpha=0.4)
    plt.legend()
    plt.title(titolo)
    plt.show()

def disegna_frontiera(modello, X, y, titolo=""):
    """Colora le regioni del piano secondo la cantina predetta dal modello (frontiera di decisione)."""
    xx, yy = np.meshgrid(np.linspace(11.9, 15.0, 300), np.linspace(3.0, 13.5, 300))
    griglia = pd.DataFrame({"alcol": xx.ravel(), "colore": yy.ravel()})
    z = np.array([1 if p == "B" else 0 for p in predici(modello, griglia)]).reshape(xx.shape)
    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, z, alpha=0.25, cmap="coolwarm_r", levels=[-0.5, 0.5, 1.5])
    for cantina, colore in COLORI.items():
        sel = (y == cantina).values
        plt.scatter(X["alcol"][sel], X["colore"][sel], c=colore, label=f"cantina {cantina}",
                    edgecolors="k", s=50)
    plt.xlabel("alcol (%)")
    plt.ylabel("intensità del colore")
    plt.legend()
    plt.title(titolo)
    plt.show()

print("✅ Funzioni pronte: predici, mostra_dati, disegna_frontiera")

In [ ]:
mostra_dati(X_train, y_train, "Training set")

### 2.3 ✏️ Esercizio: il TUO albero di decisione
Guarda il grafico: dove stanno i vini rossi (A) e dove quelli blu (B)? Scrivi un albero di decisione **con al massimo due livelli di `if`**, scegliendo tu le **soglie** (valori numerici) sulle due feature. Gli operatori di confronto sono `>`, `<`, `>=` e `<=`.

In [ ]:
def mio_albero(alcol, colore):
    # ✏️ scegli le soglie guardando il grafico del training set
    if colore > 0:
        return "B"
    else:
        return "A"

pred_mio_train = predici(mio_albero, X_train)
stampa_metriche("Il mio albero – TRAINING SET", y_train.tolist(), pred_mio_train, positivo="B")
disegna_frontiera(mio_albero, X_train, y_train, "Il mio albero")

In [ ]:
#@title 💡 Soluzione 2.3 — doppio clic per leggerla, ▶ per usarla (sostituisce il tuo codice!)
# Le soglie non sono "giuste" o "sbagliate": ognuno ne troverà di diverse. Confrontate in aula
# chi ha l'accuracy più alta sul training set… e poi vediamo chi vince sul test set!
def mio_albero(alcol, colore):
    if colore > 7.5:          # colore molto intenso → quasi sempre cantina B
        return "B"
    else:
        if alcol < 13:        # colore chiaro: i vini meno alcolici sono soprattutto della B
            return "B"
        else:
            return "A"

pred_mio_train = predici(mio_albero, X_train)
stampa_metriche("Il mio albero – TRAINING SET", y_train.tolist(), pred_mio_train, positivo="B")
disegna_frontiera(mio_albero, X_train, y_train, "Il mio albero")

### 2.4 L'albero addestrato da scikit-learn
In scikit-learn **tutti i modelli** si usano con gli stessi tre passi:
1. **creo** il modello, scegliendo i suoi **iperparametri** (le impostazioni decise da noi prima dell'addestramento);
2. **`.fit(X, y)`**: è la **fase di apprendimento (training)**, in cui il modello costruisce sé stesso a partire dai dati;
3. **`.predict(X)`**: è la **fase di inferenza**, in cui il modello predice l'output per dei nuovi input.

Per un albero, l'iperparametro più importante è **`max_depth`**, la profondità massima (quanti `if` uno dentro l'altro). Partiamo con `max_depth=2`, come il tuo.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

albero = DecisionTreeClassifier(max_depth=2, random_state=0)   # 1) creo il modello
albero.fit(X_train, y_train)                                   # 2) training
pred_sklearn_train = albero.predict(X_train)                   # 3) inferenza (qui sul training set)

stampa_metriche("sklearn max_depth=2 – TRAINING SET", y_train.tolist(), list(pred_sklearn_train), positivo="B")

In [ ]:
# plot_tree disegna l'albero appreso
plt.figure(figsize=(12, 6))      # dimensione della figura in pollici (larghezza, altezza)
plot_tree(albero,
          feature_names=["alcol", "colore"],
          class_names=list(albero.classes_),   # nomi delle classi, nell'ordine usato dal modello
          filled=True,                          # colora i nodi in base alla classe prevalente
          rounded=True)
plt.show()

**Come si legge un nodo:**
- la **prima riga** è la condizione: se è vera si va a **sinistra**, altrimenti a destra;
- **`gini`** misura l'*impurità* del nodo: vale 0 se tutti i vini del nodo sono della stessa cantina, e 0.5 se sono metà e metà. L'algoritmo sceglie le soglie che rendono i nodi figli il più "puri" possibile;
- **`samples`** è il numero di vini del training set che arrivano in quel nodo;
- **`value`** è quanti di quei vini sono della cantina A e quanti della B;
- **`class`** è la cantina predetta (la maggioranza).

E lo stesso albero, scritto come `if`/`else`:

In [ ]:
# export_text stampa l'albero come regole testuali: confrontale con il tuo mio_albero!
print(export_text(albero, feature_names=["alcol", "colore"]))

In [ ]:
disegna_frontiera(albero, X_train, y_train, "sklearn, max_depth=2")

> 🤔 Le soglie trovate da scikit-learn somigliano alle tue? La differenza fondamentale è che tu le hai scelte **a occhio**, mentre l'algoritmo le ha cercate **in modo automatico**, provando tutte le soglie possibili e scegliendo le migliori. Con 2 feature lo puoi fare anche tu; con 13 feature (o 1000) no.

### 2.5 Un esempio di overfitting: l'albero senza limiti
Finora l'albero poteva fare al massimo **2 domande di fila**. Proviamo a togliere il limite: scikit-learn continuerà ad aggiungere domande finché non avrà classificato correttamente **tutti** i vini del training set.

In [ ]:
albero_libero = DecisionTreeClassifier(random_state=0)   # senza max_depth: nessun limite al numero di domande
albero_libero.fit(X_train, y_train)

print("Domande di fila (al massimo):", albero_libero.get_depth(), "| Foglie:", albero_libero.get_n_leaves())
disegna_frontiera(albero_libero, X_train, y_train, "Albero senza limiti")

> 🤔 Guarda la frontiera di decisione: ci sono strisce sottili e "isolette" create apposta per **singoli vini**. Sul training set questo albero non sbaglia mai. **Cosa ti ricorda?** (suggerimento: Parte 1.6)

### 2.6 La sfida finale sul test set
Ora valutiamo i tre alberi su vini che **nessuno ha mai visto**.

In [ ]:
modelli = {
    "Il mio albero": mio_albero,
    "sklearn max_depth=2": albero,
    "sklearn senza limiti": albero_libero,
}

righe = []
for nome, modello in modelli.items():
    righe.append({
        "modello": nome,
        "accuracy TRAINING": accuracy_score(y_train, predici(modello, X_train)),
        "accuracy TEST": accuracy_score(y_test, predici(modello, X_test)),
    })

pd.DataFrame(righe).round(2)     # creo un DataFrame da una lista di dizionari, per una tabella leggibile

### 📌 Come si legge la tabella
- L'albero **senza limiti** è perfetto sul training set (accuracy 1.00) ma sul test set crolla (circa 0.67): ha **memorizzato** i singoli vini invece di imparare una regola generale. È lo stesso **overfitting** del memorizzatore della Parte 1.
- Gli alberi **semplici** (il tuo e quello con `max_depth=2`) sbagliano qualcosa sul training set, ma sui vini nuovi vanno molto meglio: hanno imparato una regola che **generalizza**.
- La morale: un modello più complicato **non** è automaticamente migliore. Conta quanto è bravo sui dati che non ha mai visto.

🤔 *Osserva anche*: il test set ha solo 33 vini, quindi una differenza di accuracy di 0.03 corrisponde a **un solo vino**. Con pochi dati, piccole differenze tra due modelli non dicono molto.